# Predict Tags on 5 Test Dataset Samples
### Simple Tag Prediction: Multi-Class RF-DETR vs Old `model.plan`

This notebook loads **5 sample images from the test dataset split** and predicts tags using:
1. **Multi-Class RF-DETR** (`.pth` checkpoint)
2. **Old Model** (`model.plan` TensorRT engine)

In [ ]:
# 1. Imports & Device Setup
import os
import json
import cv2
import torch
import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import torchvision.transforms.functional as TF

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


In [ ]:
# 2. Load 5 Sample Images from the TEST Dataset Split
TEST_DIR = "multi_class_train_rfdetr/dataset_full_data/test"
if not os.path.exists(TEST_DIR):
    TEST_DIR = "multi_class_train_rfdetr/dataset_sample_1000/test"

IMAGES_DIR = "multi_class_train_rfdetr/images"
ANN_FILE = os.path.join(TEST_DIR, "_annotations.coco.json")

CLASSES = ["blue_aisle", "blue_bay", "location_tag"]
test_images = []

if os.path.exists(ANN_FILE):
    with open(ANN_FILE) as f:
        coco = json.load(f)
    CLASSES = [c["name"] for c in sorted(coco.get("categories", []), key=lambda x: x["id"])]
    test_images = [os.path.join(IMAGES_DIR, im["file_name"]) for im in coco.get("images", [])[:5]]
elif os.path.exists(TEST_DIR):
    test_images = [os.path.join(TEST_DIR, f) for f in os.listdir(TEST_DIR) if f.lower().endswith((".jpg", ".png"))][:5]
else:
    # Fallback to test paths
    test_images = [f"test_image_{i}.jpg" for i in range(1, 6)]

print(f"Categories: {CLASSES}")
print(f"Selected 5 Test Images:")
for idx, path in enumerate(test_images, 1):
    print(f"  {idx}. {path} (exists: {os.path.exists(path)})")


In [ ]:
# 3. Load Models (Multi-Class RF-DETR & model.plan)
# --- Multi-Class RF-DETR ---
from rfdetr import RFDETRBase
from rfdetr.models.lwdetr import LWDETR

# Disable automatic download of coco pretrain
RFDETRBase.maybe_download_pretrain_weights = lambda self: None
RFDETRBase.load_pretrain_weights = lambda self: None

# Match weights safely
def _safe_load(self, state_dict, strict=True):
    cur = self.state_dict()
    filtered = {k.replace("model.", "").replace("module.", ""): v for k, v in state_dict.items()}
    filtered = {k: v for k, v in filtered.items() if k in cur and cur[k].shape == v.shape}
    return torch.nn.Module.load_state_dict(self, filtered, strict=False)
LWDETR.load_state_dict = _safe_load

rf_ckpt = "multi_class_train_rfdetr/model/best_model_full_data.pth"
if not os.path.exists(rf_ckpt):
    rf_ckpt = "multi_class_train_rfdetr/model/best_model_sample_1000.pth"

# Instantiate wrapper and extract the underlying PyTorch nn.Module (LWDETR)
wrapper = RFDETRBase(num_classes=len(CLASSES), resolution=1008, pretrain_weights=None)
if hasattr(wrapper, "model") and hasattr(wrapper.model, "model") and hasattr(wrapper.model.model, "load_state_dict"):
    rfdetr = wrapper.model.model
elif hasattr(wrapper, "model") and hasattr(wrapper.model, "load_state_dict"):
    rfdetr = wrapper.model
else:
    rfdetr = wrapper

if os.path.exists(rf_ckpt):
    ckpt = torch.load(rf_ckpt, map_location="cpu", weights_only=False)
    state = ckpt.get("model", ckpt)
    rfdetr.load_state_dict(state, strict=False)
    print(f"Loaded RF-DETR checkpoint: {rf_ckpt}")
else:
    print(f"Notice: Checkpoint not found at {rf_ckpt}")

rfdetr.to(device).eval()

# --- Old Model (model.plan) ---
trt_engine, trt_context = None, None
try:
    import tensorrt as trt
    plan_file = "model.plan" if os.path.exists("model.plan") else "location_tag_text_det.engine"
    if os.path.exists(plan_file):
        with open(plan_file, "rb") as f, trt.Runtime(trt.Logger(trt.Logger.WARNING)) as r:
            trt_engine = r.deserialize_cuda_engine(f.read())
            trt_context = trt_engine.create_execution_context()
        print(f"Loaded {plan_file}")
    else:
        print("model.plan file not found yet.")
except Exception as e:
    print("TensorRT note:", e)


In [ ]:
# 4. Simple Prediction Functions
CONF_THRESH = 0.50

def predict_tags_rfdetr(img_path):
    """Predicts multi-class tags using RF-DETR."""
    if not os.path.exists(img_path):
        return []
    img = Image.open(img_path).convert("RGB")
    orig_w, orig_h = img.size
    
    # Resize to 1008x1008 and normalize
    x = TF.to_tensor(img.resize((1008, 1008))).unsqueeze(0)
    x = TF.normalize(x, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]).to(device)
    
    with torch.no_grad():
        out = rfdetr(x)
        
    scores, classes = out["pred_logits"][0].sigmoid().max(-1)
    boxes = out["pred_boxes"][0]
    keep = scores > CONF_THRESH
    
    preds = []
    for s, c, (cx, cy, w, h) in zip(scores[keep], classes[keep], boxes[keep]):
        x1 = float(max(0, (cx - w / 2) * orig_w))
        y1 = float(max(0, (cy - h / 2) * orig_h))
        x2 = float(min(orig_w, (cx + w / 2) * orig_w))
        y2 = float(min(orig_h, (cy + h / 2) * orig_h))
        tag_name = CLASSES[int(c)] if int(c) < len(CLASSES) else f"class_{int(c)}"
        preds.append({"tag": tag_name, "score": round(float(s), 2), "box": [round(x1, 1), round(y1, 1), round(x2, 1), round(y2, 1)]})
    return preds


def predict_tags_plan(img_path):
    """Predicts location tags using old model.plan."""
    if trt_context is None or not os.path.exists(img_path):
        return []
    cv_img = cv2.imread(img_path)
    if cv_img is None:
        return []
    orig_h, orig_w = cv_img.shape[:2]
    
    inp = cv2.resize(cv_img, (640, 640))
    inp = torch.from_numpy(cv2.cvtColor(inp, cv2.COLOR_BGR2RGB)).permute(2, 0, 1).float().div(255.0).unsqueeze(0).to("cuda")
    
    out = torch.empty((1, 1, 640, 640), dtype=torch.float32, device="cuda")
    in_name = trt_engine.get_tensor_name(0) if hasattr(trt_engine, "get_tensor_name") else trt_engine.get_binding_name(0)
    out_name = trt_engine.get_tensor_name(1) if hasattr(trt_engine, "get_tensor_name") else trt_engine.get_binding_name(1)
    
    trt_context.set_tensor_address(in_name, inp.data_ptr())
    trt_context.set_tensor_address(out_name, out.data_ptr())
    trt_context.execute_async_v3(torch.cuda.current_stream().cuda_stream)
    torch.cuda.synchronize()
    
    mask = (out[0, 0].sigmoid() > 0.3).cpu().numpy().astype(np.uint8)
    contours, _ = cv2.findContours(mask, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    preds = []
    for cnt in contours:
        bx, by, bw, bh = cv2.boundingRect(cnt)
        if bw >= 4 and bh >= 4:
            preds.append({
                "tag": "location_tag",
                "score": round(float(out[0, 0, by:by+bh, bx:bx+bw].sigmoid().mean().item()), 2),
                "box": [round(bx * orig_w / 640, 1), round(by * orig_h / 640, 1), round((bx + bw) * orig_w / 640, 1), round((by + bh) * orig_h / 640, 1)]
            })
    return preds


In [ ]:
# 5. Run Prediction on the 5 Test Images & Print Detected Tags
for i, path in enumerate(test_images, 1):
    print(f"\n==================== Test Image {i}: {os.path.basename(path)} ====================")
    if not os.path.exists(path):
        print(f"Image file not found: {path}")
        continue
        
    rf_tags = predict_tags_rfdetr(path)
    plan_tags = predict_tags_plan(path)
    
    print(f"Multi-Class RF-DETR ({len(rf_tags)} tags detected):")
    for t in rf_tags:
        print(f"   - {t['tag']:<15} (confidence: {t['score']:.2f}) -> Box: {t['box']}")
        
    print(f"Old Model (model.plan) ({len(plan_tags)} tags detected):")
    for t in plan_tags:
        print(f"   - {t['tag']:<15} (confidence: {t['score']:.2f}) -> Box: {t['box']}")


In [ ]:
# 6. Display Predicted Tags on the 5 Test Images
COLORS = {"blue_aisle": "deepskyblue", "blue_bay": "orange", "location_tag": "lime"}

def draw_boxes(img_path, tags):
    im = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(im)
    for t in tags:
        color = COLORS.get(t["tag"], "red")
        draw.rectangle(t["box"], outline=color, width=3)
        draw.text((t["box"][0], max(0, t["box"][1] - 16)), f"{t['tag']} {t['score']}", fill=color)
    return im

valid_images = [p for p in test_images if os.path.exists(p)]
if valid_images:
    fig, axes = plt.subplots(len(valid_images), 2, figsize=(14, 4 * len(valid_images)))
    if len(valid_images) == 1:
        axes = np.array([axes])
        
    for i, p in enumerate(valid_images):
        # Left: Old model.plan
        axes[i, 0].imshow(draw_boxes(p, predict_tags_plan(p)))
        axes[i, 0].set_title(f"Test Image {i+1}: Old model.plan", fontsize=10)
        axes[i, 0].axis("off")
        
        # Right: Multi-Class RF-DETR
        axes[i, 1].imshow(draw_boxes(p, predict_tags_rfdetr(p)))
        axes[i, 1].set_title(f"Test Image {i+1}: Multi-Class RF-DETR", fontsize=10, fontweight="bold")
        axes[i, 1].axis("off")
        
    plt.tight_layout()
    plt.show()
else:
    print("Place your test images in the path above to view visual bounding boxes.")
